In [1]:
# Configurar el path para importar los módulos
import sys
import os

# Añadir el directorio raíz del proyecto al path
root_dir = os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..')))
if root_dir not in sys.path:
    sys.path.append(root_dir)

In [2]:
# Importar las funciones necesarias
from db.conexion_db import crear_conexion, cerrar_conexion
from db.querys_db import insert_data_polars, consult_data, update_data
import datetime
import polars as pl


In [3]:
import numpy as np

In [4]:
ruta_archivos_base = r'C:\Users\CGM\Projects\Proyecto CGMExPress\data'
formato = ".xlsx"
nombre_archivo_entregas_ewm_cabezera = "MONITOR_CABEZERA"
nombre_archivo_entregas_ewm_detalle = "MONITOR_DETALLE"
nombre_archivo_guias_remision = "guia_remision"
centros = pl.DataFrame({'centro': ['C200', 'C040', 'C080']})

In [5]:
rename_columns_entregas_ewm_cabezera = {
    "Documento": "id_entrega",
    "Destinatario mcía.": "codigo_cliente",
    "Descripción destinatario de mercancías": "descripcion",
    "Clase de documento": "clase_documento",
    "Oficina expedición": "oficina_expedicion",
    "Status salida de mercancías": "status_salida_mercancia",
    "Concluido": "concluido",
    "Status de picking": "status_picking",
    "Autor": "autor",
    "Cantidad pos.": "cantidad_posiciones",
    "Cantidad de unidades de manipulación": "cantidad_unidades_manipuladas",
    "Cantidad de productos": "cantidad_productos",
    "Fecha de entrega (planificada)": "Fecha_entrega_planif",
    "Creados el": "created_at_sap",
    "Creados a la(s)": "created_to_the_sap",
}

In [6]:
excel_entregas_ewm_cabezeraC154 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_cabezera+formato),engine="calamine")
excel_entregas_ewm_cabezeraC200 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_cabezera+centros["centro"][0]+formato),engine="calamine")
excel_entregas_ewm_cabezeraC040 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_cabezera+centros["centro"][1]+formato),engine="calamine")
excel_entregas_ewm_cabezeraC080 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_cabezera+centros["centro"][2]+formato),engine="calamine")

In [7]:
df_excel_entregas_ewm_cabezera = pl.concat([excel_entregas_ewm_cabezeraC154, excel_entregas_ewm_cabezeraC200, excel_entregas_ewm_cabezeraC040, excel_entregas_ewm_cabezeraC080])


In [8]:
df_excel_entregas_ewm_cabezera = df_excel_entregas_ewm_cabezera.rename(rename_columns_entregas_ewm_cabezera)

In [9]:
connection = crear_conexion("db_almacen_distribucion")
cliente_db = consult_data(connection, "SELECT id_cliente,codigo_cliente FROM tbl_cliente")

df_cliente_db = pl.DataFrame(cliente_db)

Conexion Exitosa


In [10]:
df_cliente_excel = df_excel_entregas_ewm_cabezera.select(["codigo_cliente","descripcion"]).unique()

In [11]:
df_clientes_no_encontrados = df_cliente_excel.join(df_cliente_db, on="codigo_cliente", how="anti")

In [12]:
cabezera_clientes = ["codigo_cliente","descripcion"]
batch_size = 1000

In [ ]:
if not df_clientes_no_encontrados.empty:
    con = crear_conexion("db_almacen_distribucion")
    insert_data_polars(con, "tbl_cliente", df_clientes_no_encontrados, cabezera_clientes, batch_size)
    cerrar_conexion(con)
else:   
    print('Dataframe en blanco')

Conexion Exitosa
✅ Se insertaron: 69 registros en la tabla tbl_cliente (Polars)
conexion cerrada


In [25]:
configuracion_tipos_guias = {
                                "Status": pl.String,
                                "Sociedad": pl.String, 
                                "DocumSAP": pl.String,
                                "Ejercicio": pl.String,
                                "TipDocumento": pl.String,
                                "Número SUNAT": pl.String,
                                "Estado del Proceso": pl.String,
                                "Fecha de Creación": pl.String,
                                "Hora Creación": pl.String,
                                "Usuario Creación": pl.String,
                                "Fecha emisión": pl.String,
                                "Nombres y/o Razón Social del Adquiriente": pl.String,
                                "N° CDR": pl.String,
                                "Fecha Recepción": pl.String,
                                "Hora Recepción": pl.String,
                                "Fecha Respuesta": pl.String, 
                                "Hora Respuesta": pl.String, 
                                "Mensaje": pl.String,
                                "Observación": pl.String}

In [52]:
excel_guias = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_guias_remision+formato),engine="calamine", schema_overrides=configuracion_tipos_guias)

In [65]:
df_guias = excel_guias.select([ "DocumSAP", "Número SUNAT", "Fecha de Creación", "Hora Creación", "Usuario Creación", "Fecha emisión"])

In [66]:
rename_columns_guias = {"DocumSAP": "entrega", 
                       "Número SUNAT": "guia_sap", 
                       "Fecha de Creación": "created_at_sap",
                       "Hora Creación": "created_to_the_sap",
                       "Usuario Creación": "usuario", 
                       "Fecha emisión": "fecha_emision", }
df_guias = df_guias.rename(rename_columns_guias)

In [67]:
df_guias = df_guias.with_columns(
    pl.col("created_at_sap")
      .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False),

    pl.col("created_to_the_sap")
      .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False),

    pl.col("fecha_emision")
      .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False)
)

df_guias = df_guias.with_columns(
    pl.col("created_at_sap").dt.date().alias("created_at_sap"),
    pl.col("created_to_the_sap").dt.time().alias("created_to_the_sap"),
    pl.col("fecha_emision").dt.date().alias("fecha_emision")
)

In [68]:
df_guias_unicas = df_guias.sort(
    ["entrega", "created_at_sap", "created_to_the_sap"]
).unique(
    subset=["entrega"], 
    keep="last"
)

In [ ]:
cabezera_guias = [ "guia_sap","entrega", "usuario", "fecha_emision", "created_at_sap", "created_to_the_sap"]

In [ ]:
connection = crear_conexion("db_almacen_distribucion")
insert_data_polars(connection, "tbl_guia_remision_zosfe004", df_guias_unicas,cabezera_guias, batch_size) 
cerrar_conexion(connection)

Conexion Exitosa
✅ Se insertaron: 346 registros en la tabla tbl_guia_remision_zosfe004 (Polars)


NameError: name 'con' is not defined

In [ ]:
connection = crear_conexion("db_almacen_distribucion")
cliente_db = consult_data(connection, "SELECT id_cliente,codigo_cliente FROM tbl_cliente")
centro_db = consult_data(connection, "SELECT id_centro, cod_centro FROM tbl_centros")
guiasap_db = consult_data(connection, "SELECT id_guia_remision, guia_sap,entrega FROM tbl_guia_remision_zosfe004")
df_cliente_db = pl.DataFrame(cliente_db)
df_centro_db = pl.DataFrame(centro_db)
cerrar_conexion(connection)

Conexion Exitosa


In [ ]:
df_entregas_join = df_excel_entregas_ewm_cabezera.join(df_cliente_db, on="codigo_cliente", how="left")

In [ ]:
df_entregas_to_mysql_finalizadas = df_entregas_join_cliente.select(["id_entrega","id_cliente","clase_documento","oficina_expedicion","status_salida_mercancia","concluido","status_picking","cantidad_posiciones","cantidad_unidades_manipuladas","cantidad_productos","autor","Fecha_entrega_planif","created_at_sap","created_to_the_sap"]).filter(pl.col("status_salida_mercancia") == "Finalizada")

In [ ]:
df_entregas_to_mysql_pendientes = df_entregas_join_cliente.select(["id_entrega","id_cliente","clase_documento","oficina_expedicion","status_salida_mercancia","concluido","status_picking","cantidad_posiciones","cantidad_unidades_manipuladas","cantidad_productos","autor","Fecha_entrega_planif","created_at_sap","created_to_the_sap"]).filter(pl.col("status_salida_mercancia") != "Finalizada")

In [ ]:
cabezera_entregas = ["id_entrega","id_cliente","clase_documento","oficina_expedicion","status_salida_mercancia","concluido",
                     "status_picking","cantidad_posiciones","cantidad_unidades_manipuladas","cantidad_productos","autor",
                     "Fecha_entrega_planif","created_at_sap","created_to_the_sap"]
batch_size = 1000

In [ ]:
con = crear_conexion("db_almacen_distribucion")
insert_data_polars(con, "tbl_entrega_ewm", df_entregas_to_mysql_finalizadas, cabezera_entregas, batch_size)
cerrar_conexion(con)

Conexion Exitosa
✅ Se insertaron: 93 registros en la tabla tbl_entrega_ewm (Polars)
conexion cerrada


In [ ]:
con = crear_conexion("db_almacen_distribucion")
insert_data_polars(con, "tbl_entrega_pendientes_ewm", df_entregas_to_mysql_pendientes, cabezera_entregas, batch_size)
cerrar_conexion(con)

Conexion Exitosa
✅ Se insertaron: 5 registros en la tabla tbl_entrega_pendientes_ewm (Polars)
conexion cerrada


In [ ]:
configuracion_tipos = {"Documento": pl.String, 
                       "Número de posición": pl.String, 
                       "Pedido": pl.String, 
                       "Pedido clte.": pl.String, 
                       "Número de reserva": pl.String, 
                       "Orden de mantenimiento": pl.String,
                       "Prod.": pl.String, 
                       "Ctd.": pl.Float64, 
                       "Unidad de medida": pl.String, 
                       "Lote": pl.String, 
                       "Fecha de picking real": pl.String, 
                       "Hora de picking real": pl.String}   

In [ ]:
rename_columns_entregas_ewm_detalle = {"Documento": "id_entrega", 
                       "Número de posición": "posicion", 
                       "Pedido": "pedido", 
                       "Pedido clte.": "pedido_venta", 
                       "Número de reserva": "reserva", 
                       "Orden de mantenimiento": "orden",
                       "Prod.": "codigo_material", 
                       "Ctd.": "cantidad", 
                       "Unidad de medida": "unidad_medida", 
                       "Lote": "lote", 
                       "Fecha de picking real": "fecha_picking", 
                       "Hora de picking real": "hora_picking"}

In [ ]:
excel_entregas_ewm_detalleC154 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_detalle+formato),engine="calamine",schema_overrides=configuracion_tipos)
excel_entregas_ewm_detalleC200 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_detalle+centros["centro"][0]+formato),engine="calamine",schema_overrides=configuracion_tipos)
excel_entregas_ewm_detalleC040 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_detalle+centros["centro"][1]+formato),engine="calamine",schema_overrides=configuracion_tipos)
excel_entregas_ewm_detalleC080 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_detalle+centros["centro"][2]+formato),engine="calamine",schema_overrides=configuracion_tipos)

In [ ]:
df_excel_entregas_ewm_detalle = pl.concat([excel_entregas_ewm_detalleC154, excel_entregas_ewm_detalleC200, excel_entregas_ewm_detalleC040, excel_entregas_ewm_detalleC080])

In [ ]:
df_excel_entregas_ewm_detalle = df_excel_entregas_ewm_detalle.with_columns(
    
    # Convertir Fecha
    pl.col("Fecha de picking real").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).dt.date().alias("Fecha de picking real"),
    
    # Convertir Hora
    # Formato común SAP/Excel: "%H:%M:%S" (ej: 14:30:59)
    pl.col("Hora de picking real").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).dt.time().alias("Hora de picking real")
).rename(rename_columns_entregas_ewm_detalle)

In [ ]:
df_materiales_excel = pl.read_excel(r'C:\Users\CGM\Projects\Proyecto CGMExPress\maestros\materiales.xlsx', engine="calamine")

In [ ]:
connection = crear_conexion("db_matricial")
materiales_db = consult_data(connection, "SELECT id_material,codigo_material FROM tbl_materiales_ih09")
df_material_db = pl.DataFrame(materiales_db)    

Conexion Exitosa
conexion cerrada


In [ ]:
connection = crear_conexion("db_matricial")
tipo_material = consult_data(connection, "SELECT id_tipo_material,codigo_tipo_material FROM tbl_tipo_material")
df_tipo_material_db = pl.DataFrame(tipo_material)

Conexion Exitosa
conexion cerrada


In [ ]:
connection = crear_conexion("db_matricial")
grupo_articulo_db = consult_data(connection, "SELECT id_grupo_articulo,codigo_grupo_articulo FROM tbl_grupo_articulo")
df_grupo_articulo_db = pl.DataFrame(grupo_articulo_db)

Conexion Exitosa
conexion cerrada


In [ ]:
df_materiales_no_encontrados = df_materiales_excel.join(df_material_db, right_on="codigo_material", left_on="Material", how="anti")

In [ ]:
rename_columns_materiales = {
    "Material": "codigo_material",
    "Texto breve de material": "descripcion",
    "Tipo material": "tipo_material",
    "Grupo de artículos": "grupo_articulo",
    "Nº antiguo material": "codigo_antiguo",
    "Unidad medida base": "unidad_medida_base",
    "Peso bruto": "peso_bruto",
    "Peso neto": "peso_neto",
    "Unidad de peso": "unidad_peso",
    "Volumen": "volumen",
    "Unidad de volumen": "unidad_volumen",
    "Jerarquía productos": "jerarquia",
    "Nº pieza fabricante": "numero_pieza",
    "Código EAN/UPC": "codigo_ean",
    "Creado por": "created_by_sap",
    "Creado el": "created_at_sap",
}

In [ ]:
df_materiales_no_encontrados = df_materiales_no_encontrados.rename(rename_columns_materiales)

In [ ]:
df_materiales_join = df_materiales_no_encontrados.join(df_tipo_material_db, left_on="tipo_material", right_on="codigo_tipo_material", how="left").join(df_grupo_articulo_db, left_on="grupo_articulo", right_on="codigo_grupo_articulo", how="left").select(['codigo_material','descripcion','id_tipo_material','id_grupo_articulo','codigo_antiguo',
                       'unidad_medida_base','peso_bruto','peso_neto','unidad_peso','volumen','unidad_volumen',
                       'jerarquia','numero_pieza','codigo_ean','created_by_sap','created_at_sap'])

In [ ]:
cabezera_materiales = ['codigo_material','descripcion','id_tipo_material','id_grupo_articulo','codigo_antiguo',
                       'unidad_medida_base','peso_bruto','peso_neto','unidad_peso','volumen','unidad_volumen',
                       'jerarquia','numero_pieza','codigo_ean','created_by_sap','created_at_sap']

In [ ]:
connection = crear_conexion("db_matricial")
insert_data_polars(connection, "tbl_materiales_ih09", df_materiales_join,cabezera_materiales, batch_size)
cerrar_conexion(con)

Conexion Exitosa
✅ Se insertaron: 18 registros en la tabla tbl_materiales_ih09 (Polars)


In [ ]:
connection = crear_conexion("db_matricial")
materiales_db = consult_data(connection, "SELECT id_material,codigo_material FROM tbl_materiales_ih09")
df_material_db = pl.DataFrame(materiales_db)

Conexion Exitosa
conexion cerrada


In [ ]:
df_excel_entregas_ewm_detalle_join = df_excel_entregas_ewm_detalle.join(df_material_db, left_on="codigo_material", right_on="codigo_material", how="left").select(["id_entrega","posicion","pedido","pedido_venta","reserva","orden","id_material","cantidad","unidad_medida","lote","fecha_picking","hora_picking"])

In [ ]:
df_excel_entregas_ewm_detalle_finalizados = df_excel_entregas_ewm_detalle_join.join(df_entregas_to_mysql_finalizadas.select(["id_entrega"]), on="id_entrega", how="inner")

In [ ]:
cabezera_entregas_detalle = ["id_entrega","posicion","pedido","pedido_venta","reserva","orden","id_material","cantidad","unidad_medida","lote","fecha_picking","hora_picking"]

In [ ]:
connection = crear_conexion("db_almacen_distribucion")
insert_data_polars(connection, "tbl_detalle_entrega_ewm", df_excel_entregas_ewm_detalle_finalizados,cabezera_entregas_detalle, batch_size)
cerrar_conexion(con)

Conexion Exitosa
✅ Se insertaron: 431 registros en la tabla tbl_detalle_entrega_ewm (Polars)
